# 📊 CIDRS Model Evaluation & Comparison
**Date:** May 25, 2026

## Models Evaluated:
1. **Random Forest** - Baseline ensemble method
2. **LSTM** - Deep learning for sequential patterns
3. **Autoencoder** - Anomaly detection for zero-day threats

## Evaluation Metrics:
- Accuracy, Precision, Recall, F1-Score
- ROC-AUC, False Positive Rate, False Negative Rate
- Inference Time

In [ ]:
# Import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json
import joblib
import tensorflow as tf
import time

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("✅ Libraries imported")

In [ ]:
# Load model metrics
with open('models/random_forest/metrics.json', 'r') as f:
    rf_metrics = json.load(f)

with open('models/lstm/metrics.json', 'r') as f:
    lstm_metrics = json.load(f)

with open('models/autoencoder/metrics.json', 'r') as f:
    ae_metrics = json.load(f)

print("✅ Model metrics loaded")

In [ ]:
# Create comparison dataframe
comparison_data = {
    'Model': ['Random Forest', 'LSTM', 'Autoencoder'],
    'Accuracy': [rf_metrics['accuracy'], lstm_metrics['accuracy'], ae_metrics['accuracy']],
    'Precision': [rf_metrics['precision'], lstm_metrics['precision'], ae_metrics['precision']],
    'Recall': [rf_metrics['recall'], lstm_metrics['recall'], ae_metrics['recall']],
    'F1-Score': [rf_metrics['f1_score'], lstm_metrics['f1_score'], ae_metrics['f1_score']],
    'ROC-AUC': [rf_metrics['roc_auc'], lstm_metrics['roc_auc'], ae_metrics['roc_auc']],
    'False Positive Rate': [rf_metrics['false_positive_rate'], lstm_metrics['false_positive_rate'], ae_metrics['false_positive_rate']],
    'False Negative Rate': [rf_metrics['false_negative_rate'], lstm_metrics['false_negative_rate'], ae_metrics['false_negative_rate']]
}

df_comparison = pd.DataFrame(comparison_data)
df_comparison

## 1. Performance Bar Chart

In [ ]:
# Bar chart comparison
fig, ax = plt.subplots(figsize=(12, 6))

metrics_to_plot = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
x = np.arange(len(metrics_to_plot))
width = 0.25

rf_values = [df_comparison[metric].iloc[0] * 100 for metric in metrics_to_plot]
lstm_values = [df_comparison[metric].iloc[1] * 100 for metric in metrics_to_plot]
ae_values = [df_comparison[metric].iloc[2] * 100 for metric in metrics_to_plot]

bars1 = ax.bar(x - width, rf_values, width, label='Random Forest', color='#3498db')
bars2 = ax.bar(x, lstm_values, width, label='LSTM', color='#2ecc71')
bars3 = ax.bar(x + width, ae_values, width, label='Autoencoder', color='#e74c3c')

ax.set_ylabel('Score (%)', fontsize=12)
ax.set_title('Model Performance Comparison', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(metrics_to_plot)
ax.legend(loc='lower right')
ax.set_ylim(90, 100)

# Add value labels on bars
for bars in [bars1, bars2, bars3]:
    for bar in bars:
        height = bar.get_height()
        ax.annotate(f'{height:.1f}%',
                   xy=(bar.get_x() + bar.get_width() / 2, height),
                   xytext=(0, 3),
                   textcoords="offset points",
                   ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig('../docs/model_comparison_bars.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Chart saved: docs/model_comparison_bars.png")

## 2. Radar Chart (Multi-metric Comparison)

In [ ]:
# Radar chart
from math import pi

metrics_radar = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC']
rf_radar = [df_comparison[m].iloc[0] * 100 for m in metrics_radar]
lstm_radar = [df_comparison[m].iloc[1] * 100 for m in metrics_radar]
ae_radar = [df_comparison[m].iloc[2] * 100 for m in metrics_radar]

# Number of variables
N = len(metrics_radar)
angles = [n / float(N) * 2 * pi for n in range(N)]
angles += angles[:1]

# Initialize radar chart
fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(projection='polar'))

# Plot each model
rf_radar += rf_radar[:1]
lstm_radar += lstm_radar[:1]
ae_radar += ae_radar[:1]

ax.plot(angles, rf_radar, 'o-', linewidth=2, label='Random Forest', color='#3498db')
ax.fill(angles, rf_radar, alpha=0.25, color='#3498db')

ax.plot(angles, lstm_radar, 'o-', linewidth=2, label='LSTM', color='#2ecc71')
ax.fill(angles, lstm_radar, alpha=0.25, color='#2ecc71')

ax.plot(angles, ae_radar, 'o-', linewidth=2, label='Autoencoder', color='#e74c3c')
ax.fill(angles, ae_radar, alpha=0.25, color='#e74c3c')

# Add labels
ax.set_xticks(angles[:-1])
ax.set_xticklabels(metrics_radar, fontsize=10)
ax.set_ylim(90, 100)
ax.set_title('Model Performance Radar Chart', fontsize=14, fontweight='bold', pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.0))

plt.tight_layout()
plt.savefig('../docs/model_comparison_radar.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Chart saved: docs/model_comparison_radar.png")

## 3. Error Rates Comparison

In [ ]:
# Error rates bar chart
fig, ax = plt.subplots(figsize=(10, 6))

error_metrics = ['False Positive Rate', 'False Negative Rate']
x = np.arange(len(error_metrics))
width = 0.25

rf_errors = [df_comparison[metric].iloc[0] * 100 for metric in error_metrics]
lstm_errors = [df_comparison[metric].iloc[1] * 100 for metric in error_metrics]
ae_errors = [df_comparison[metric].iloc[2] * 100 for metric in error_metrics]

ax.bar(x - width, rf_errors, width, label='Random Forest', color='#3498db')
ax.bar(x, lstm_errors, width, label='LSTM', color='#2ecc71')
ax.bar(x + width, ae_errors, width, label='Autoencoder', color='#e74c3c')

ax.set_ylabel('Error Rate (%)', fontsize=12)
ax.set_title('Error Rates Comparison (Lower is Better)', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(error_metrics)
ax.legend()

plt.tight_layout()
plt.savefig('../docs/error_rates_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Chart saved: docs/error_rates_comparison.png")

## 4. Inference Time Benchmark

In [ ]:
# Measure inference time
import time

# Load models
rf_model = joblib.load('../models/random_forest/rf_model.pkl')
lstm_model = tf.keras.models.load_model('../models/lstm/lstm_model.h5')
ae_model = tf.keras.models.load_model('../models/autoencoder/autoencoder.h5')

# Create test sample
sample = np.random.rand(1, 41)

# Benchmark Random Forest
start = time.time()
for _ in range(1000):
    rf_model.predict(sample)
rf_time = (time.time() - start) / 1000 * 1000  # ms per prediction

# Benchmark LSTM
start = time.time()
for _ in range(100):
    lstm_model.predict(sample, verbose=0)
lstm_time = (time.time() - start) / 100 * 1000  # ms per prediction

# Benchmark Autoencoder
start = time.time()
for _ in range(100):
    ae_model.predict(sample, verbose=0)
ae_time = (time.time() - start) / 100 * 1000  # ms per prediction

# Create bar chart
fig, ax = plt.subplots(figsize=(10, 6))

models = ['Random Forest', 'LSTM', 'Autoencoder']
times = [rf_time, lstm_time, ae_time]
colors = ['#3498db', '#2ecc71', '#e74c3c']

bars = ax.bar(models, times, color=colors)
ax.set_ylabel('Inference Time (ms)', fontsize=12)
ax.set_title('Model Inference Time Comparison (Lower is Better)', fontsize=14, fontweight='bold')
ax.axhline(y=200, color='red', linestyle='--', label='Target: <200ms')

# Add value labels
for bar, time in zip(bars, times):
    ax.annotate(f'{time:.1f} ms',
               xy=(bar.get_x() + bar.get_width() / 2, bar.get_height()),
               xytext=(0, 3),
               textcoords="offset points",
               ha='center', va='bottom')

ax.legend()
plt.tight_layout()
plt.savefig('../docs/inference_time_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Chart saved: docs/inference_time_comparison.png")

## 5. Final Summary Table

In [ ]:
# Format for thesis
summary_table = df_comparison.copy()
for col in summary_table.columns:
    if col != 'Model':
        summary_table[col] = (summary_table[col] * 100).round(2).astype(str) + '%'

print("=" * 80)
print("📊 CIDRS MODEL PERFORMANCE SUMMARY")
print("=" * 80)
print(summary_table.to_string(index=False))

# Save to CSV for thesis
summary_table.to_csv('../docs/model_performance_summary.csv', index=False)
print("\n✅ Summary saved to: docs/model_performance_summary.csv")

## 6. Key Insights

In [ ]:
print("=" * 60)
print("📋 KEY INSIGHTS")
print("=" * 60)

insights = f"""
┌─────────────────────────────────────────────────────────────┐
│                    MODEL COMPARISON RESULTS                  │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│  🏆 BEST ACCURACY:     LSTM ({lstm_metrics['accuracy']*100:.2f}%)                     │
│  🏆 BEST PRECISION:    LSTM ({lstm_metrics['precision']*100:.2f}%)                     │
│  🏆 BEST RECALL:       LSTM ({lstm_metrics['recall']*100:.2f}%)                       │
│  🏆 BEST F1-SCORE:     LSTM ({lstm_metrics['f1_score']*100:.2f}%)                     │
│  🏆 FASTEST INFERENCE: Random Forest ({rf_time:.1f}ms)                  │
│                                                             │
├─────────────────────────────────────────────────────────────┤
│                    ENSEMBLE RECOMMENDATION                   │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│  ✅ Use Random Forest for real-time, high-speed detection  │
│  ✅ Use LSTM for maximum accuracy on complex attacks       │
│  ✅ Use Autoencoder for zero-day/anomaly detection         │
│  ✅ Final system uses ALL THREE in ensemble voting         │
│                                                             │
└─────────────────────────────────────────────────────────────┘
"""
print(insights)

# Save insights
with open('../docs/model_evaluation_insights.txt', 'w') as f:
    f.write(insights)
print("✅ Insights saved to: docs/model_evaluation_insights.txt")